In [ ]:
#| default_exp dojo

# dojo

> Practice katas for solveit dialog tooling, scored on the route taken, not just the outcome

Agents reproduce whatever patterns their context shows them, so rather than only documenting the dialog tooling's best routes (`view_dlg` to orient, `msg_lnhashview` then `msg_exhash` to edit, id-threaded `add_msg` to insert), the dojo makes a session *perform* each one once, scored, before real work starts. `await dojo_start()` seeds a small practice dialog and prints a card of katas with a par for the round; every cell and tool call afterwards is a stroke on the ledger, and `dojo_score()` replays it -- per-kata strokes vs par, habit findings, and each kata's checker verdict. A clean round issues a completion id that future sessions can pass to `dojo_start` to skip the gate. `dojo_bench` runs the same gate against a list of models, each in a fresh dialog.

In [ ]:
#| export
"Practice katas for solveit dialog tooling, scored on the route taken, not just the outcome. Start with `await dojo_start()`."
import ast,json,re,time,uuid,asyncio
from fastcore.utils import *
from dialoghelper.core import *
from dialoghelper.core import _add_msg_unsafe
from dialoghelper.exhash import *
from pyskills import allow

In [ ]:
#| hide
from fastcore.test import *

## Strokes

Solveit runs every AI tool call as a kernel cell of the form `await call_tool(f, {...})` and every code message as a plain cell, so both transports score uniformly: the round hooks `pre_run_cell` and each costly cell is one stroke. Reading docs is free, so there is no incentive to skip them.

In [ ]:
#| export
def _splitmagic(src):
    "Split a leading `%%magic` line from `src`, returning `(magic_name, body)`"
    if not src.startswith('%%'): return None, src
    hd, _, body = src.partition('\n')
    return hd[2:].split()[0], body

def _parse(src):
    try: return ast.parse(src)
    except SyntaxError: return None

def _calls(tree):
    for n in ast.walk(tree):
        if isinstance(n, ast.Call): yield n

def _callee(c): return c.func.id if isinstance(c.func, ast.Name) else None

def _fname(v):
    "Called name of (possibly awaited) call `v`, resolving `call_tool(f, ...)` to `f`"
    if isinstance(v, ast.Await): v = v.value
    if not isinstance(v, ast.Call): return None
    nm = _callee(v)
    if nm=='call_tool' and v.args and isinstance(v.args[0], ast.Name): return v.args[0].id
    return nm

In [ ]:
t = ast.parse("await call_tool(view_dlg, {'dname': '/dojo_runs/x'})")
test_eq(_fname(t.body[0].value), 'view_dlg')
test_eq(_fname(ast.parse("doc(msg_exhash)").body[0].value), 'doc')

In [ ]:
#| export
_MACHINERY = {'dojo_start','dojo_score','dojo_redo','dojo_resume','dojo_kata','dojo_report','forget_dojo'}
_FREE = _MACHINERY | {'doc','doced','forget_doced','help','list_pyskills'}
_MACH_WRAP = {'print','dumps'}
_MAGIC_COST = {'bash': 2}

def _is_free(src):
    "A cell costs nothing if it only reads docs, imports, or comments -- as code or as tool calls"
    magic, body = _splitmagic(src)
    if magic: return False
    tree = _parse(body)
    if tree is None: return False
    def _ok(n):
        if isinstance(n, (ast.Import, ast.ImportFrom)): return True
        if isinstance(n, ast.Expr): return _fname(n.value) in _FREE
        if isinstance(n, ast.For) and not any(_calls(n.iter)): return all(map(_ok, n.body))
        return False
    return all(map(_ok, tree.body))

def _is_machinery(src):
    "Pure dojo_* cells (optionally import- or print/json.dumps-wrapped) stay out of the trace, so rescoring never grows the ledger"
    tree = _parse(_splitmagic(src)[1])
    if tree is None or not tree.body: return False
    if not all(isinstance(n, (ast.Import, ast.ImportFrom, ast.Expr)) for n in tree.body): return False
    nms = {_fname(c) or (c.func.attr if isinstance(c.func, ast.Attribute) else None) for c in _calls(tree)}
    return bool(nms & _MACHINERY) and nms <= _MACHINERY | _MACH_WRAP

def _cost(src):
    if _is_free(src): return 0
    return _MAGIC_COST.get(_splitmagic(src)[0], 1)

In [ ]:
for s,c in [("await call_tool(view_dlg, {'dname': '/d'})", 1), ("await call_tool(doc, {'sym': 'exhash.skill'})", 0),
             ("# a comment\n# another", 0), ("import httpx", 0), ("%%bash\nls", 2), ("await dojo_kata(2)", 0),
             ("for f in (doc, help): doc(f)", 0), ("x = 1", 1)]:
    test_eq(_cost(s), c)
for s,e in [("dojo_score(orient='x')", True), ("await call_tool(dojo_score, {'orient': 'x'})", True),
            ("import json\njson.dumps(await dojo_report())", True), ("doc(msg_exhash)", False), ("r = await dojo_report()", False)]:
    test_eq(_is_machinery(s), e)

## The practice dialog

`dojo_start` seeds a fresh dialog from `_SEED`: a tiny "weather" project whose messages are the kata targets. Checkers are pure functions over the dialog's `(msgs, ids)` so they can be tested without a running solveit.

In [ ]:
#| export
_SQ3 = "'''"   # can't appear literally inside the rf''' below: the same trap kata 3 sets
_TMPL_PAYLOAD = rf'''def render(name, temp):
    r"""Render a one-line summary; keep \t, \n and {_SQ3} literal in this docstring."""
    return name + ':\t' + str(temp) + ' degrees\n'
'''
_TMPL_IND = '\n'.join('    '+l for l in _TMPL_PAYLOAD.splitlines())

In [ ]:
#| export
_SEED = [
    ('title', 'note', '# Weather daily\n\nFetches daily weather from the open-meteo API and renders one-line summaries.'),
    ('setup', 'code', 'import httpx'),
    ('config', 'code', '''DEFAULTS = dict(units='imperial', lang='en')

def load_cfg(path):
    "Read a cfg file from `path`, merging entries over `DEFAULTS`"
    cfg = dict(DEFAULTS)
    for line in open(path).read().splitlines():
        k, v = line.split('=', 1)
        cfg[k.strip()] = v.strip()
    return cfg'''),
    ('fetch', 'code', '''def fetch_daily(lat, lon):
    "Fetch one day of weather for `lat`,`lon`"
    r = httpx.get('https://api.open-meteo.com/v1/forecast',
                  params=dict(latitude=lat, longitude=lon, daily='temperature_2m_max'))
    return r.json()['daily']'''),
    ('tmpl', 'code', '''TMPL_VERSION = 1

def render(name, temp):
    # OLD_TMPL: verbose builder kept from the prototype
    parts = []
    parts.append(name)
    parts.append(': ')
    parts.append(str(temp))
    parts.append(' degrees')
    out = ''.join(parts)
    out = out + '.'
    return out'''),
    ('retries_h', 'note', '## Retries'),
    ('retries', 'note', 'On a connection error, `fetch_daily` retries the request twice before giving up.'),
    ('example', 'code', "d = httpx.get('https://api.open-meteo.com/v1/forecast?latitude=52&longitude=0&daily=temperature_2m_max').json()\nlist(d)"),
]
_CHANGELOG = ['## Changelog', '- v0.1: first release', '- v0.2: metric units', '- v0.3: retry logic']

def _seed_content(key): return next(c for k,mt,c in _SEED if k==key)

In [ ]:
#| export
def _content(msgs, id): return next((m['content'] for m in msgs if m['id']==id), '')

def _chk_orient(msgs, ids, orient=''):
    a = orient or ''
    if not a: return ['no answer passed: dojo_score(orient="<your prose answer>")']
    out = []
    if not ('meteo' in a.lower() or ('fetch' in a.lower() and 'daily' in a.lower())):
        out.append('answer does not describe fetching daily open-meteo weather: read the dialog, not just one message')
    if (missing := [ids[k] for k in ('fetch','example') if ids[k] not in a]):
        out.append(f"answer does not name httpx-calling message(s) {', '.join(missing)} by id")
    return out

def _chk_config(msgs, ids, orient=''):
    t = _content(msgs, ids['config'])
    out = []
    if "units='metric'" not in t or 'imperial' in t: out.append("default units is not 'metric'")
    if len(re.findall(r'\bcfg\b', t)) != 1 or len(re.findall(r'\bconfig\b', t)) != 3:
        out.append('cfg -> config rename incomplete, or the docstring was changed')
    return out

def _chk_tmpl(msgs, ids, orient=''):
    t = _content(msgs, ids['tmpl'])
    out = []
    if 'TMPL_VERSION = 1' not in t: out.append('the TMPL_VERSION line was not kept')
    if 'OLD_TMPL' in t: out.append('old render() body still present')
    if _TMPL_PAYLOAD.strip() not in t: out.append('replacement render() does not match the provided text verbatim')
    if '\n\n\n\n' in t: out.append('stray blank lines left around the replacement')
    return out

def _chk_retries(msgs, ids, orient=''):
    t = _content(msgs, ids['retries'])
    return [] if '3 attempts' in t and 'twice before giving up' not in t else \
        ['the Retries note does not say "3 attempts"']

def _chk_changelog(msgs, ids, orient=''):
    i = next((i for i,m in enumerate(msgs) if m['id']==ids['title']), -1)
    got = [m['content'].strip() for m in msgs[i+1:i+5]]
    if got != _CHANGELOG: return [f'the four notes after the title are not the Changelog in order (found: {got[:2]}...)']
    if any(m['msg_type']!='note' for m in msgs[i+1:i+5]): return ['the Changelog messages are not note messages']
    return []

In [ ]:
#| export
KATAS = [
    dict(name='orient', par=1, check=_chk_orient, ro=True,
        route='one view_dlg(dname=...) shows every message with its id; answer from the bare result',
        prompt='What does the practice dialog build, and which code messages call httpx? Answer in prose, naming messages by id, and pass it via dojo_score(orient="...").'),
    dict(name='edit set', par=2, keys=['config'], check=_chk_config,
        route='msg_lnhashview, then ONE msg_exhash with all commands, worked bottom-to-top',
        prompt="In the config message {config}: change the default units to 'metric', and rename cfg to config everywhere in code (whole-word matches; the docstring stays unchanged; load_cfg may stay or be renamed)."),
    dict(name='hostile replace', par=2, keys=['tmpl'], check=_chk_tmpl,
        route='msg_lnhashview, then one msg_exhash with a range-c command whose payload lands verbatim ("%" c would replace the whole message: too much here)',
        prompt='In the template message {tmpl}: replace the whole render() function (keeping the TMPL_VERSION line) with exactly this, verbatim:\n\n' + _TMPL_IND),
    dict(name='note fix', par=2, keys=['retries'], check=_chk_retries,
        route="find_msgs(header_section='## Retries', dname=...) -- the exact markdown header line, ## included -- then one update_msg(id, content=...): a whole-note rewrite needs no line addresses",
        prompt='The markdown note under the Retries header is wrong; it should say the request is retried twice more, making "3 attempts" in all.'),
    dict(name='ordered insert', par=2, ins=True, check=_chk_changelog,
        route="ONE code cell threading ids -- mid = await add_msg(s, id=mid, dname=...) per note; without id, add_msg places after the *current* message, so un-threaded calls land in REVERSE",
        prompt="Insert notes '## Changelog', '- v0.1: first release', '- v0.2: metric units', '- v0.3: retry logic' -- in that order, immediately after the title note {title}."),
]

In [ ]:
#| hide
import copy
ids = {k: f'_{k}' for k,_,_ in _SEED}
msgs0 = [dict(id=f'_{k}', msg_type=mt, content=c) for k,mt,c in _SEED]
for f in (_chk_config, _chk_tmpl, _chk_retries, _chk_changelog): assert f(msgs0, ids), f.__name__

def _set(msgs, id, c): next(m for m in msgs if m['id']==id)['content'] = c
msgs1 = copy.deepcopy(msgs0)
_set(msgs1, '_config', _seed_content('config').replace("'imperial'","'metric'").replace('cfg = dict','config = dict')
     .replace('cfg[k','config[k').replace('return cfg','return config'))
_set(msgs1, '_tmpl', 'TMPL_VERSION = 1\n\n' + _TMPL_PAYLOAD.strip())
_set(msgs1, '_retries', 'On a connection error, `fetch_daily` retries the request twice more, making 3 attempts in all.')
msgs1[1:1] = [dict(id=f'_new{i}', msg_type='note', content=c) for i,c in enumerate(_CHANGELOG)]
a = 'Fetches daily open-meteo weather; httpx called in _fetch and _example'
for f in (_chk_orient, _chk_config, _chk_tmpl, _chk_retries, _chk_changelog): test_eq(f(msgs1, ids, a), [])

msgs2 = copy.deepcopy(msgs1)             # load_cfg -> load_config also accepted
_set(msgs2, '_config', _content(msgs1,'_config').replace('load_cfg','load_config'))
test_eq(_chk_config(msgs2, ids), [])
msgs3 = copy.deepcopy(msgs1)             # docstring damage still caught
_set(msgs3, '_config', _content(msgs1,'_config').replace('a cfg file','a config file'))
assert _chk_config(msgs3, ids)

## Kata tags

Strokes attribute to katas by the *current* tag: `dojo_kata(n)` (free machinery, callable as a tool) or a leading `# kata <n>:` comment in a code cell; later cells inherit it until the next tag.

In [ ]:
#| export
_TAG_RE = re.compile(r'katas?\W{0,3}(\d[\d\s,+&/-]*)', re.I)

def _kata_tag(src):
    "Kata number from a leading `# kata <n>:` comment; only text before the first ':' counts, and the last mention wins"
    for l in src.splitlines():
        l = l.strip()
        if not l: continue
        if not l.startswith('#'): break
        ns = [int(n) for m in _TAG_RE.finditer(l.split(':',1)[0]) for n in re.findall(r'\d+', m.group(1))]
        if (v := [n for n in ns if 1<=n<=len(KATAS)]): return v[-1]
    return None

In [ ]:
for s,t in [("# kata 2: rename cfg\nx=1", 2), ("# kata 1+4: shared", 4), ("x=1\n# kata 3:", None),
            ("# kata 2 done, kata 3 next: onwards", 3), ("# kata 9:", None), ("#kata2:", 2), ("# note: kata 4 stuff", None)]:
    test_eq(_kata_tag(s), t)

## Findings

Habit rules replayed over the ledger at scoring time. Each finding blocks a clean round; the doc penalty is forgiven if the docs were read by scoring time.

In [ ]:
#| export
_HASH_RE = re.compile(r'\d+\|[0-9a-f]{4}\|')

def _lit(x):
    if x is None: return None
    try: return ast.literal_eval(x)
    except (ValueError, TypeError, SyntaxError): return None

def _msgcall(v):
    "`(name, id, cmds)` for a (possibly awaited or `call_tool`'d) call, with literal args resolved"
    if isinstance(v, ast.Await): v = v.value
    if not isinstance(v, ast.Call): return None, None, None
    nm, args, kws = _callee(v), v.args, {k.arg: k.value for k in v.keywords}
    if nm=='call_tool' and args and isinstance(args[0], ast.Name):
        d = _lit(args[1]) if len(args)>1 else {}
        d = d if isinstance(d, dict) else {}
        return args[0].id, d.get('id'), d.get('cmds')
    return nm, _lit(args[0] if args else kws.get('id')), _lit(args[1] if len(args)>1 else kws.get('cmds'))

def _hashaddrs(cmds):
    "Do any of `cmds` use `lineno|hash|` addresses?"
    return any(isinstance(c,(list,tuple)) and c and isinstance(c[0],str) and _HASH_RE.search(c[0]) for c in cmds)

def _scan(entries):
    "Habit findings and doc penalties from the trace, replayed in order"
    finds, pen, viewed, doced = {}, [], set(), False
    for e in entries:
        tree = _parse(_splitmagic(e['src'])[1])
        if tree is None: continue
        for c in _calls(tree):
            nm, id, cmds = _msgcall(c)
            if nm in ('lnhash','line_hash'):
                finds['hashcalc'] = 'exhash addresses come only from a fresh msg_lnhashview; never compute them'
            if nm in ('doc','doced') and 'exhash' in e['src']: doced = True
            if nm in ('msg_lnhashview','msg_exhash') and not doced: pen.append(nm)
            if nm=='msg_lnhashview' and id: viewed.add(id)
            if nm=='msg_exhash' and id:
                if cmds and _hashaddrs(cmds) and id not in viewed:
                    finds['stale_view'] = 'Always msg_lnhashview a message afresh before a hash-addressed msg_exhash: past views go stale'
                viewed.discard(id)
    return finds, (['msg_lnhashview/msg_exhash used before doc(exhash.skill)'] if pen and not doced else [])

In [ ]:
good = [dict(src="doc(exhash.skill)"),
        dict(src="await call_tool(msg_lnhashview, {'id': '_m1', 'dname': '/dojo_runs/x'})"),
        dict(src="await call_tool(msg_exhash, {'id': '_m1', 'cmds': [['3|ab12|', 's', 'cfg', 'config']], 'dname': '/dojo_runs/x'})")]
test_eq(_scan(good), ({}, []))
assert _scan(good[1:])[1]                                     # no doc() at all -> penalty
test_eq(_scan(good[1:] + [good[0]]), ({}, []))                # doc'd by scoring time -> forgiven
assert 'stale_view' in _scan([good[0], good[2]])[0]           # hash edit without a view
assert 'stale_view' in _scan(good + [dict(src=good[2]['src'] + ' ')])[0]   # edit invalidates the view
assert 'hashcalc' in _scan([dict(src="addr = lnhash(3, 'x')")])[0]

## Running a round

`dojo_start` seeds the dialog and registers the `pre_run_cell` tracer; `dojo_score` replays the ledger; `dojo_redo` resets one kata, *replacing* its strokes rather than adding to them. A clean round removes the practice dialog and records a completion id (valid one week, per tooling version).

In [ ]:
#| export
_RUN, _LAST = {}, None
_WEEK = 7*86400

def _version():
    import dialoghelper
    return getattr(dialoghelper, '__version__', '0')

def _complete_file(): return Path.home()/'.dialoghelper_dojo.json'
def _last_file(): return Path.home()/'.dialoghelper_dojo_last.json'

def _write_last(d):
    "Persist the latest score to disk so a harness can read it after the dialog's kernel is gone"
    _last_file().write_text(json.dumps(d))

def _completions():
    "Completion records `{id: {t, v}}` from the last week; older entries are pruned"
    f = _complete_file()
    recs = json.loads(f.read_text()) if f.exists() else {}
    return {k:r for k,r in recs.items() if r['t'] > time.time()-_WEEK}

def forget_dojo():
    "Truncate the dojo completion record (e.g. after a tooling change): every session redoes the round"
    _complete_file().write_text('{}')

In [ ]:
#| export
def _log(info):
    "pre_run_cell hook: ledger each cell; machinery and exact resends stay out (kernel hiccups are not your route)"
    if _RUN.get('paused') or _is_machinery(info.raw_cell): return
    src, tr = info.raw_cell, _RUN['trace']
    if tr and tr[-1]['src']==src: return
    if (t := _kata_tag(src)): _RUN['kata'] = t
    tr.append(dict(src=src, kata=_RUN.get('kata')))

def _strokes(entries):
    "`(untagged, per_kata)` stroke totals from the trace"
    unt, per = 0, [0]*len(KATAS)
    for e in entries:
        if not (c := _cost(e['src'])): continue
        if e.get('kata'): per[e['kata']-1] += c
        else: unt += c
    return unt, per

def dojo_kata(
    n:int  # Kata to attribute the following strokes to; 0 returns to untagged
):
    "Start attributing strokes to kata `n` (free, like all dojo_* machinery)"
    if not _RUN: return 'No active run: dojo_start() first.'
    assert 0 <= n <= len(KATAS), f'kata number must be 0-{len(KATAS)}'
    _RUN['kata'] = n or None
    return f"tagging kata {n}: {KATAS[n-1]['name']}" if n else "untagged"

In [ ]:
#| hide
_RUN.update(trace=[], kata=None)
class _I: raw_cell = ''
def _cell(src):
    i = _I(); i.raw_cell = src; _log(i)
_cell("dojo_kata(1)"); dojo_kata(1)   # the tag cell executes after its (skipped) log
for s in ["await call_tool(view_dlg, {'dname': '/d'})", "# kata 2: edits\nx=1", "x=1", "x=1", "%%bash\nls"]:
    _cell(s)
test_eq(_strokes(_RUN['trace']), (0, [1, 4, 0, 0, 0]))   # dup 'x=1' resend free; %%bash=2 inherits kata 2
_RUN.clear()

In [ ]:
#| export
def _fmt(s):
    "Fill `{key}` message-id references from the seeded dialog"
    for k, v in _RUN['ids'].items(): s = s.replace('{'+k+'}', v)
    return s

def _card():
    d = _RUN['dname']
    ks = '\n'.join(f"{i}. (par {k['par']}) {_fmt(k['prompt'])}" for i,k in enumerate(KATAS,1))
    return f"""== solveit dojo ==
Practice dialog: {d} -- address it with dname='{d}' in EVERY dialoghelper call (your own dialog and kernel are otherwise unchanged, and all tooling is preloaded: no imports needed).
Tag your work, starting with your first cell: call dojo_kata(<n>) (free, works as a tool call too), or start a code cell with a free '# kata <n>:' comment; later cells inherit the tag, and orientation work belongs to kata 1. dojo_redo(0) discards accidental untagged strokes.
Scoring: every cell you run and every tool call = 1 stroke; a %%bash cell = 2 (dialog work never needs the shell). Free: an exact resend of the previous cell, cells of only doc()/help/list_pyskills()/imports/comments, and all dojo_* machinery -- as code or as tool calls.
Penalties: +1 if msg_lnhashview/msg_exhash is used before reading doc(exhash.skill).
Findings (each blocks a clean round): computing lnhash addresses by hand; a hash-addressed msg_exhash on a message not msg_lnhashview'd since its last edit.
Par assumes the tooling's best route, shown with each kata at scoring: matching par means you found it.
Par for the round: {sum(k['par'] for k in KATAS)}. When done: dojo_score(orient="<your kata-1 answer>").
The round is complete ONLY on a clean score: par or better, every kata ok, no penalties, no findings. Until then do no work outside the dojo; redo over-par katas with dojo_redo(n), in ascending order. Scoring pauses the ledger: dojo_redo (or dojo_resume() without a reset) restarts it.
This dojo is an early version: note anything about the scoring or process that seems possibly-imperfect, and include it in your report.

{ks}"""

In [ ]:
#| export
async def dojo_start(
    id:str=None  # Completion id from a previous clean round: skips the gate when on record
):
    "Set up a fresh practice run: seed a private dialog, start tracing, and print the kata card."
    if id:
        rec = _completions().get(id)
        if rec and rec.get('v')==_version(): return print(f'Dojo already complete (id {id}): no tasks.')
        print(f'id {id!r} not on record: expired, truncated, or the tooling changed since. Run the round.')
    from IPython import get_ipython
    d = f'/dojo_runs/{uuid.uuid4().hex[:8]}'
    await create_or_run_dialog(d, template=False)
    ids = {}
    for k, mt, c in _SEED:
        mid = await add_msg(c, msg_type=mt, dname=d, placement='at_end')
        assert mid.startswith('_'), f'seeding failed: {mid}'
        ids[k] = mid
    ip = get_ipython()
    if _RUN.get('ip'):   # a prior unfinished round: drop its hook and practice dialog so nothing stale leaks in
        try: _RUN['ip'].events.unregister('pre_run_cell', _log)
        except ValueError: pass
        await _rm_run(_RUN['dname'])
    _RUN.clear()
    _RUN.update(dname=d, ids=ids, trace=[], ip=ip, kata=None)
    ip.events.register('pre_run_cell', _log)
    print(_card())

In [ ]:
#| export
async def _rm_run(d):
    "The one place we delete a dialog: refuse anything not strictly under /dojo_runs/"
    assert d.startswith('/dojo_runs/') and len(d)>len('/dojo_runs/'), f'refusing to delete {d}'
    await stop_dialog(d)
    await rm_dialog(d)

async def _score(orient=''):
    "Everything `dojo_score` prints and `dojo_report` returns, as one dict"
    msgs = await find_msgs(dname=_RUN['dname'])
    ids, tr = _RUN['ids'], _RUN['trace']
    unt, per = _strokes(tr)
    finds, pens = _scan(tr)
    ks = [dict(name=k['name'], strokes=s, par=k['par'], probs=k['check'](msgs, ids, orient), route=k['route'])
          for k, s in zip(KATAS, per)]
    strokes, pen, par = unt+sum(per), len(pens), sum(k['par'] for k in KATAS)
    ok = not finds and not pen and strokes+pen<=par and not any(k['probs'] for k in ks)
    return dict(strokes=strokes, untagged=unt, pen=pen, pens=pens, par=par, katas=ks,
                finds=finds, ok=ok, tagged=any(e.get('kata') for e in tr))

In [ ]:
#| export
async def dojo_score(
    orient:str=''  # Your kata-1 prose answer
):
    "Score the run: strokes vs par, habit findings from the trace, and each kata's outcome."
    global _LAST
    if not _RUN: return print('No active run: dojo_start() first.')
    _RUN['orient'] = orient
    r = _LAST = await _score(orient)
    _write_last({**r, 'dname': _RUN['dname'], 't': time.time(), 'orient': orient})
    print(f"strokes {r['strokes']:g} + doc penalties {r['pen']} = {r['strokes']+r['pen']:g}, par {r['par']}")
    for e in _RUN['trace']: print(f"  {_cost(e['src'])}| {(e['src'].splitlines() or [''])[0][:70]}")
    if r['pens']: print(f"  doc penalty: {'; '.join(r['pens'])}")
    for name, note in r['finds'].items(): print(f'habit miss [{name}]: {note}')
    overs = [i for i, k in enumerate(r['katas'], 1) if r['tagged'] and k['strokes'] > k['par']]
    for k in r['katas']:
        lbl = f" (strokes {k['strokes']:g}, par {k['par']})" if r['tagged'] else ''
        print(f"kata '{k['name']}'{lbl}: {'; '.join(k['probs']) if k['probs'] else 'ok'}\n  par route: {k['route']}")
    if overs and not r['ok']: print('over-par katas: ' + ', '.join(f'dojo_redo({i})' for i in overs) + ' resets and retries them, in ascending order')
    if r['tagged'] and r['untagged']: print(f"{r['untagged']:g} untagged strokes: tag with dojo_kata(<n>) or a '# kata <n>:' comment, or dojo_redo(0) to discard accidental ones")
    elif not r['tagged']: print("For per-kata scoring, tag work with dojo_kata(<n>) or a leading '# kata <n>:' comment; later cells inherit it.")
    if r['strokes']+r['pen'] > r['par']: print(f"{r['strokes']+r['pen']-r['par']:g} over par for the round: replay with the par routes in mind")
    if r['ok']: await _finish()
    else:
        _RUN['paused'] = True
        print('Fix the misses, then dojo_redo(<kata number>) to reset that kata and try again (ascending order when several). Ledger paused: dojo_redo or dojo_resume() restarts it; the round completes only on a clean score.')
    print('The dojo is an early version: include in your report anything above that seemed possibly-imperfect (stroke counts, findings, prompts).')

async def _finish():
    "Clean round: unhook the tracer, remove the practice dialog, and record a completion id"
    try: _RUN['ip'].events.unregister('pre_run_cell', _log)
    except ValueError: pass
    await _rm_run(_RUN['dname'])
    _RUN.clear()
    cid = uuid.uuid4().hex[:4]
    recs = _completions(); recs[cid] = dict(t=time.time(), v=_version())
    _complete_file().write_text(json.dumps(recs))
    print(f'Clean round. Practice dialog removed. Completion id: {cid} -- keep this id, including through compaction: passing dojo_start({cid!r}) in a future session skips the round.')

In [ ]:
#| export
async def dojo_redo(
    n:int  # Kata to reset and retry; 0 resets no kata and discards accidental untagged strokes
):
    "Reset kata `n` for a fresh try: pristine messages, its strokes dropped from the trace (the retry replaces them), and the ledger resumed."
    if not _RUN: return print('No active run: dojo_start() first.')
    _RUN['paused'] = False
    k = KATAS[n-1] if n else None
    _RUN['trace'] = [e for e in _RUN['trace'] if not (_cost(e['src']) and (e.get('kata')==n if n else not e.get('kata')))]
    if not k: return print('Untagged strokes discarded; ledger resumed.')
    _RUN['kata'] = n
    d, ids = _RUN['dname'], _RUN['ids']
    if k.get('ins'):
        for m in await find_msgs(dname=d):
            if m['id'] not in ids.values(): await del_msg(m['id'], dname=d)
    elif not k.get('ro'):
        for key in k['keys']: await update_msg(ids[key], content=_seed_content(key), dname=d)
    p = _fmt(k['prompt']).splitlines()[0] + (' ...' if '\n' in k['prompt'] else '')
    print(f"kata '{k['name']}' reset. Par {k['par']}: {p}")

def dojo_resume():
    "Resume stroke counting after a mid-round dojo_score, without resetting any kata"
    if not _RUN: return print('No active run: dojo_start() first.')
    _RUN['paused'] = False
    print('Ledger resumed: counted work continues.')

async def dojo_report(
    orient:str=''  # Kata-1 answer when scoring fresh; defaults to the last one passed
)->dict:
    "Machine-readable score for harnesses: the active round's fresh score, or the last completed one"
    if not _RUN: return dict(active=False) | (_LAST or {})
    r = await _score(orient or _RUN.get('orient',''))
    return dict(active=True, dname=_RUN['dname']) | r | \
        dict(katas=[{k2:v2 for k2,v2 in k.items() if k2!='route'} for k in r['katas']])

## Benchmarks

Testing the gate against models: `dojo_bench` flips solveit's standard-model setting (a global setting, so runs are sequential and `restore=` puts yours back), runs the gate prompt in a fresh dialog per model, and pulls `dojo_report()` out of that dialog's kernel.

In [ ]:
#| export
import httpx

def _read_last(after, dname=None):
    "The persisted score if it was written after `after` (else None): survives the dialog kernel dying"
    f = _last_file()
    if not f.exists(): return None
    d = json.loads(f.read_text())
    return d if d.get('t', 0) > after else None

async def _run_state(id, dname):
    "Message run-state, retrying transient resets from the busy single-process server"
    for _ in range(5):
        try: return (await read_msgid(id, dname=dname)).get('run')
        except httpx.TransportError: await asyncio.sleep(1)
    return (await read_msgid(id, dname=dname)).get('run')

async def _await_turn(id, dname, timeout, poll=2.):
    "Wait until prompt `id` in `dname` finishes its turn"
    t0 = time.time()
    while time.time()-t0 < timeout:
        if not await _run_state(id, dname): return
        await asyncio.sleep(poll)
    raise TimeoutError(f'prompt {id} in {dname} still running after {timeout}s')

In [ ]:
#| export
_TOOLS = ('dojo_start dojo_kata dojo_score dojo_redo dojo_resume view_dlg find_msgs read_msgid view_msg '
          'msg_lnhashview msg_exhash add_msg update_msg del_msg doc').split()

_GATE = ('Complete the dojo warm-up round now, in this single turn, without pausing to ask or report until it is done. '
         'Call dojo_start(); it prints a card of katas. Do EVERY kata in the card by its stated rules, keeping tool calls '
         'flowing one after another -- do not stop after dojo_start. Then call dojo_score(orient="<your kata-1 answer>"). '
         'If the score is not clean, immediately dojo_redo(n) the over-par katas (ascending) and dojo_score again, and keep '
         'going until the round is clean. Only once dojo_score reports a clean round, stop and reply with the completion id.')

_CONTINUE = 'Continue the dojo round: keep calling the dojo tools until dojo_score reports a clean round, then reply with the completion id. Do not stop early.'

async def _seed_bench(d):
    "Reference the tools and import the dojo in `d`: solveit resolves &-sigil tool schemas from the kernel, so the import MUST finish before the first prompt turn"
    await add_msg('Tools for this dialog: &`[' + ', '.join(_TOOLS) + ']`', dname=d, placement='at_end')
    await _add_msg_unsafe('from dialoghelper.dojo import *\nfrom ipykernel_helper import call_tool',
                          msg_type='code', run=True, wait=True, dname=d, placement='at_end')

async def _set_model(m):
    "Point solveit's standard model at `m`"
    await call_endpa('update_model_setting_', required=False, name='standard_model', value=m)

async def _dojo_runs():
    "Names of currently-present practice dialogs"
    try: return {i.rstrip('/') for i in (await list_dialogs('/dojo_runs'))['items']}
    except Exception: return set()

async def _sweep(before):
    "Remove practice dialogs created since `before` (headless rounds may not self-clean)"
    for name in (await _dojo_runs()) - before:
        try: await _rm_run('/dojo_runs/'+name)
        except (AssertionError, httpx.TransportError): pass

async def _bench1(m, folder, timeout, turns, poll):
    "Run the gate against one model, returning its persisted score (read from disk, not the dialog kernel)"
    await _set_model(m)
    d = f"{folder}/{re.sub(r'[^A-Za-z0-9._-]', '-', m)}-{uuid.uuid4().hex[:4]}"
    await create_or_run_dialog(d, template=False)
    await _seed_bench(d)
    before, t0, rep = await _dojo_runs(), time.time(), None
    for _ in range(turns):
        pid = await add_msg(_GATE if rep is None else _CONTINUE, msg_type='prompt', run=True, dname=d, placement='at_end')
        await _await_turn(pid, d, timeout, poll)
        rep = _read_last(t0, d)
        if rep and (rep.get('ok') or not rep.get('active', True)): break
    await _sweep(before)
    return dict(dialog=d) | (rep or dict(error='no dojo_score recorded'))

In [ ]:
#| export
async def dojo_bench(
    models:list, # Model ids to test, e.g. ['anthropic/claude-sonnet-5', 'openai/gpt-5.4']
    folder:str='/dojo_bench', # Folder for the per-model bench dialogs
    restore:str=None, # Model id to restore as the standard model afterwards (there is no read endpoint)
    timeout:int=900, # Max seconds per model turn
    turns:int=4, # Max prompt turns ("Continue." after the first) per model
    poll:float=2., # Seconds between run-state polls
)->dict: # Report dict per model id, with 'dialog' pointing at the transcript
    "Run the dojo gate against each model in a fresh dialog, collecting each score from its on-disk sidecar."
    res = {}
    for m in models:
        try: res[m] = await _bench1(m, folder, timeout, turns, poll)
        except (httpx.TransportError, TimeoutError) as e: res[m] = dict(error=f'{type(e).__name__}: {e}')
    if restore: await _set_model(restore)
    return res

def bench_table(res):
    "Markdown table of `dojo_bench` results"
    hd = '| model | strokes | pen | par | clean | katas ok | finds |\n|---|---|---|---|---|---|---|'
    def _r(m, r):
        ks = r.get('katas', [])
        return (f"| {m} | {r.get('strokes','-')} | {r.get('pen','-')} | {r.get('par','-')} | {'yes' if r.get('ok') else 'NO'} "
                f"| {sum(not k['probs'] for k in ks)}/{len(ks) or '-'} | {', '.join(r.get('finds', {})) or r.get('error','-')} |")
    return '\n'.join([hd, *[_r(m,r) for m,r in res.items()]])

In [ ]:
# await dojo_start()   # seeds /dojo_runs/<id> and prints the card
# res = await dojo_bench(['anthropic/claude-sonnet-5', 'openai/gpt-5.4'], restore='anthropic/claude-fable-5')
# print(bench_table(res))

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()